### Instruksi Pengerjaan

Buat notebook baru **`Tugas6_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu bangun pipeline ETL lengkap mengikuti struktur **Extract → Transform → Load**:

In [1]:
# Impor library yang dibutuhkan
from pyspark.sql import SparkSession

# Membuat SparkSession untuk Tugas Mandiri
spark = SparkSession.builder \
    .appName("Tugas6_2505060005_Rayhan Aditya Putra") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi log yang tidak perlu agar tampilan lebih rapi
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession untuk Tugas Mandiri siap!")
print("Versi Spark:", spark.version)

26/09/24 03:14:47 WARN Utils: Your hostname, Ubuntu24 resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/24 03:14:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 03:14:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 03:14:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession untuk Tugas Mandiri siap!
Versi Spark: 3.5.9


**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.


In [2]:
# A. EXTRACT
# 1. Membaca Transaksi (CSV)
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
print("Data Transaksi:", df_transaksi.count(), "baris")
df_transaksi.printSchema()

# 2. Membaca Produk (JSON)
df_produk = spark.read.json("tugas6_produk.json")
print("Data Produk:", df_produk.count(), "baris")
df_produk.printSchema()

# 3. Membaca Ulasan (CSV)
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)
print("Data Ulasan:", df_ulasan.count(), "baris")
df_ulasan.printSchema()

Data Transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data Produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data Ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).


In [7]:
from pyspark.sql.functions import col

# B. TRANSFORM - JOIN
# Join Transaksi ke Produk menggunakan INNER JOIN (karena setiap transaksi pasti ada produknya)
df_gabung_1 = df_transaksi.join(df_produk, on="product_id", how="inner")

# Join hasilnya ke Ulasan menggunakan LEFT JOIN (karena tidak semua transaksi memiliki ulasan)
df_gabung_2 = df_gabung_1.join(df_ulasan, on="order_id", how="left")

# Menambahkan kolom total_pendapatan (unit_terjual * harga)
df_gabung_2 = df_gabung_2.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

# Menampilkan 5 baris pertama untuk mengecek hasil join
df_gabung_2.show(5)

# Mengecek total baris setelah join
print("Total baris setelah digabung:", df_gabung_2.count())

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|         1500000|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|          450000|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
o

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).


In [4]:
# C. TRANSFORM - DATA QUALITY
# 1. Tambah kolom 'ada_ulasan' (True jika rating tidak null, False jika null) SEBELUM na.fill()
df_transform = df_gabung_2.withColumn("ada_ulasan", col("rating").isNotNull())

# 2. Mengisi nilai rating yang kosong (null) dengan angka 0
df_transform = df_transform.fillna({"rating": 0})

df_transform.select("order_id", "nama_produk", "unit_terjual", "rating", "ada_ulasan", "total_pendapatan").show(5)

+--------+-----------+------------+------+----------+----------------+
|order_id|nama_produk|unit_terjual|rating|ada_ulasan|total_pendapatan|
+--------+-----------+------------+------+----------+----------------+
|     TX0|  Produk-21|           2|     1|      true|           50000|
|     TX1|   Produk-8|           6|     0|     false|         1500000|
|     TX2|  Produk-25|           4|     2|      true|          100000|
|     TX3|   Produk-3|           4|     5|      true|          300000|
|     TX4|  Produk-19|           6|     0|     false|          450000|
+--------+-----------+------------+------+----------+----------------+
only showing top 5 rows



Mengapa nilai 0 masuk akal untuk rating kosong?
Mengisi nilai kosong dengan angka 0 masuk akal karena 0 secara sistematis membedakannya dari rating paling buruk (1 bintang). Dengan nilai 0, tim analis dapat dengan mudah mengidentifikasi status "belum dinilai" tanpa merusak perhitungan rata-rata kepuasan produk (karena angka 0 dapat diabaikan atau difilter saat menghitung rata-rata skor ulasan).

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

In [5]:
# D. LOAD
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

# Membuat direktori jika belum ada (opsional tapi disarankan)
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6/

# Menyimpan ke HDFS dengan format Parquet, dipartisi berdasarkan 'kategori'
df_transform.write.mode("overwrite").partitionBy("kategori").parquet(path_hdfs)

# Memverifikasi struktur folder di HDFS
print("Struktur Folder di HDFS:")
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

# Membaca kembali dari HDFS untuk memastikan data tersimpan utuh
df_final = spark.read.parquet(path_hdfs)
print("\nJumlah baris hasil ETL yang berhasil dimuat dari HDFS:", df_final.count())

Struktur Folder di HDFS:
-rw-r--r--   3 nexvandar supergroup          0 2026-09-24 03:26 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - nexvandar supergroup          0 2026-09-24 03:26 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 nexvandar supergroup      16361 2026-09-24 03:26 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-6e47a92e-2827-46a3-8ee8-234d0ba44607.c000.snappy.parquet
drwxr-xr-x   - nexvandar supergroup          0 2026-09-24 03:26 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 nexvandar supergroup      11131 2026-09-24 03:26 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-6e47a92e-2827-46a3-8ee8-234d0ba44607.c000.snappy.parquet
drwxr-xr-x   - nexvandar supergroup          0 2026-09-24 03:26 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 nexvandar supergroup      10389 2026-09-24 03:26 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-6e47a92e-2827-46a3-8ee8-234d


Jumlah baris hasil ETL yang berhasil dimuat dari HDFS: 5000


**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?


In [6]:
from pyspark.sql.functions import avg, round

# E. INSIGHT AKHIR
# Menghitung persentase ulasan. Karena 'ada_ulasan' bernilai boolean (True=1, False=0), 
# rata-ratanya dikali 100 langsung menghasilkan persentase.
df_insight = df_final.groupBy("kategori") \
    .agg(round(avg(col("ada_ulasan").cast("int")) * 100, 2).alias("persentase_diulas")) \
    .orderBy("persentase_diulas")

df_insight.show()

[Stage 24:>                                                         (0 + 5) / 5]

+------------+-----------------+
|    kategori|persentase_diulas|
+------------+-----------------+
|     Makanan|            68.84|
|   Kesehatan|            68.89|
|     Fashion|            70.14|
|Rumah Tangga|            70.55|
|  Elektronik|            70.66|
+------------+-----------------+



Mengapa hal ini penting diketahui oleh tim marketing?
Mengetahui kategori dengan persentase ulasan terendah sangat penting bagi tim marketing untuk mengalokasikan strategi dorongan ulasan (seperti diskon, poin, atau kampanye email) secara lebih tepat sasaran. Ulasan pelanggan adalah social proof utama di e-commerce; jika suatu kategori sepi ulasan, angka konversi penjualan di kategori tersebut akan terhambat, bahkan jika kualitas produknya sangat baik.

**EKSPLORASI**

In [8]:
from pyspark.sql.functions import sum, avg, round, desc

# F. EKSPLORASI TAMBAHAN: Analisis "Hero Product"
# Mencari 5 produk dengan total pendapatan tertinggi beserta rata-rata ratingnya
df_hero_product = df_final.groupBy("nama_produk", "kategori") \
    .agg(
        sum("total_pendapatan").alias("total_pendapatan_kotor"),
        round(avg("rating"), 2).alias("rata_rata_rating")
    ) \
    .orderBy(desc("total_pendapatan_kotor")) \
    .limit(5)

df_hero_product.show()

[Stage 35:>                                                         (0 + 5) / 5]

+-----------+------------+----------------------+----------------+
|nama_produk|    kategori|total_pendapatan_kotor|rata_rata_rating|
+-----------+------------+----------------------+----------------+
|  Produk-13|     Fashion|             178750000|            2.01|
|  Produk-29|Rumah Tangga|             164250000|            2.19|
|   Produk-2|  Elektronik|             160750000|            2.26|
|  Produk-30|  Elektronik|             158500000|            2.13|
|   Produk-8|     Fashion|             155500000|            2.18|
+-----------+------------+----------------------+----------------+



Interpretasi Bisnis (Eksplorasi):
Melalui analisis Hero Product, perusahaan tidak hanya melihat produk mana yang menghasilkan uang paling banyak, tetapi juga memantau sentimen pelanggan terhadap produk andalan tersebut. Jika ada produk dengan pendapatan sangat tinggi namun rata-rata ratingnya rendah, tim pengembang produk dapat segera melakukan investigasi dan perbaikan kualitas sebelum terjadi lonjakan komplain dari pelanggan.

In [9]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
